# Link do Danych:

(https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Wczytywanie i wstępne obrabianie danych do dalszej analizy

In [ ]:
df_raw = pd.read_csv(f"penguins_lter.csv")

df_raw

In [ ]:
df_processed = df_raw.drop(
    columns="Stage\tstudyName\tRegion	Sample Number	Comments".split("\t")
).dropna()

df_processed

In [ ]:
df_processed["Species"] = df_processed.apply(lambda x: re.sub(r" [P|p]enguin[\w ()]*", "",x["Species"]), axis=1)

df_processed

In [ ]:
df_processed["Date Egg"] = pd.to_datetime(df_processed["Date Egg"])

df_processed

In [ ]:
first_variables = "Individual ID\tSpecies\tIsland\tClutch Completion\tSex\tDate Egg".split("\t")

df = df_processed[first_variables + [
    col for col in df_processed.columns if col not in set(first_variables)
]]

df = df[df["Sex"]!="."]

df

# Wizualizacja Danych (przed obróbką)

In [ ]:
from typing import Literal

In [ ]:
categorical_variables = first_variables[1:-1]

categorical_variables

In [ ]:
numerical_variables = "Culmen Length (mm)	Culmen Depth (mm)	Flipper Length (mm)	Body Mass (g)	Delta 15 N (o/oo)	Delta 13 C (o/oo)".split("\t")

numerical_variables

### Bonus: Wykresy Kropkowe (z KDE zamiast histogramów)

In [ ]:
for cat_var in categorical_variables:
    sns.pairplot(
        df,
        hue=cat_var
    )

Z wykresów kropkowych wspomaganych KDE (`Kernel Density Estimation`) widzimy że główne różnice między pingwinami przebiegają między gatunkiem `Specie` oraz płcią `Sex`.

## Zmienne Ilościowe

In [ ]:
def plot_numerical_variable(
        plot_func, 
        df:pd.DataFrame, 
        cols_included: list[str], 
        figsize_reversed: bool = False,
        hue: Literal['Species', 'Island', 'Clutch Completion', 'Sex']|None = None,
    ) -> None:
    if figsize_reversed == False:
        fig, axs = plt.subplots(len(cols_included), figsize = (12, 3*len(cols_included)))
    else:
        fig, axs = plt.subplots(1, len(cols_included), figsize = (3*len(cols_included), 12))

    for ax_idx, num_var in enumerate(cols_included):
        if plot_func != sns.histplot:
            if hue is not None:
                plot_func(ax = axs[ax_idx], data = df[[num_var, hue]], x = num_var, hue=hue)
            else:
                plot_func(ax = axs[ax_idx], data = df[num_var])
        else:
            if hue is not None:
                plot_func(ax = axs[ax_idx], data = df[[num_var, hue]], x = num_var, hue=hue, multiple="dodge", shrink = 0.9)
            else:
                plot_func(ax = axs[ax_idx], data = df[num_var], multiple="dodge", shrink = 0.9)
        axs[ax_idx].grid(True, ls = "--")

    fig.tight_layout()

### Histogramy

In [ ]:
plot_numerical_variable(
    sns.histplot,
    df,
    numerical_variables,
    False
)

### Dystrybuanty Empriczyne

In [ ]:
plot_numerical_variable(
    sns.ecdfplot,
    df, 
    numerical_variables,
    False
)

### Wykresy Wiolinowe

In [ ]:
plot_numerical_variable(
    sns.violinplot,
    df, 
    numerical_variables,
    True
)

Z Wykresów Violinowych jasno widać że rozkłady bez uwzględnienia gautnku i płci - Nie Są Normalne a wręcz są Nie-Liniowe.

### Wykresy Pudełkowe (Box-Plots)

In [ ]:
plot_numerical_variable(
    sns.boxplot,
    df,
    numerical_variables,
    True
)

## Zmienne Jakościowe

In [ ]:
def plot_categorical_variable(
        df:pd.DataFrame, 
        cols_included: list[str], 
        x_category: Literal['Species', 'Island', 'Clutch Completion', 'Sex'], 
        hue: Literal['Species', 'Island', 'Clutch Completion', 'Sex'], 
        kind: Literal['strip', 'swarm', 'box', 'violin', 'boxen', 'point', 'bar', 'count']
    ) -> None:

    for ax_idx, num_var in enumerate(cols_included):
        sns.catplot(
            data=df,
            x = x_category,
            hue=hue,
            y = num_var,
            kind=kind,
        )
        plt.grid(True, ls = "--")


### Histogram

In [ ]:
plot_numerical_variable(
    sns.histplot,
    df, 
    numerical_variables,
    hue="Species",
)

Jasno widać podział na Gatunek wśród zmiennych ilościowych.

In [ ]:
plot_numerical_variable(
    sns.histplot,
    df, 
    numerical_variables,
    hue="Sex",
)

Widzimy przesunięcie Rozkładu między Płciami

### Dytrybuanty Empiryczne

In [ ]:
plot_numerical_variable(
    sns.ecdfplot,
    df, 
    numerical_variables,
    hue="Sex"
)

In [ ]:
plot_numerical_variable(
    sns.ecdfplot,
    df, 
    numerical_variables,
    hue="Species"
)

### Violin Plots

In [ ]:
plot_categorical_variable(
    df, numerical_variables,
    x_category="Species",
    hue="Sex",
    kind="violin"
)

W sposób Bezpośredni Zaobserwowaliśmy Różnice między pingwinami na Gatunek (katoegorie wykresów wiolinowych) i płeć (Kolor Wykresu)

### Box-Plots

In [ ]:
plot_categorical_variable(
    df, numerical_variables,
    x_category="Species",
    hue="Sex",
    kind="box"
)

# Estymacja Parametrów, Przedziały Ufności, Testy na zgodność z rozkładem

## Przed podziałem względem gatunku 

In [ ]:
from scipy.stats import norm, expon, logistic, bootstrap

### Estymacja parametrów i Przedziały Ufności

In [ ]:
def calculate_dists_by_boostrap(
        df: pd.DataFrame, 
        numerical_variables: list[str], 
        dists_to_calc: dict, 
        round_to: int = 3,
        plot: bool = True,
    ) -> tuple[dict[str, dict[str, float]], dict[str, dict[str, np.ndarray]]]:

    stats_dict: dict[str, dict[str, float]] = {}
    boostrap_confidences: dict[str, dict[str, np.ndarray]] = {}

    if plot:
        fig, axs = plt.subplots(len(numerical_variables), 1, figsize = (12, 3 * len(numerical_variables)))

        fig.tight_layout()
    else:
        axs = range(len(numerical_variables))



    for ax, num_var in zip(axs, numerical_variables):
        stats_dict[num_var] = {}
        boostrap_confidences[num_var] = {}


        dists_fitted_params: dict[str, tuple] = {
            dist_name: dist["dist"].fit(df[num_var]) for dist_name, dist in dists_to_calc.items()
        }

        if plot:
            sns.histplot(df[num_var], shrink=0.9, ax = ax, stat = "density")

            dom = np.linspace(df[num_var].min(), df[num_var].max(), 10_000)
            for dist_name, dist in dists_to_calc.items():
                ax.plot(dom, dist["dist"].pdf(dom, *dists_fitted_params[dist_name]), label = dist_name)

            ax.legend()

        for dist_name, params in dists_fitted_params.items():
            stats_dict[num_var].update({
                f"{dist_name}_{param_name}": round(param, round_to) for param_name, param in zip(dists_to_calc[dist_name]["params"], params)
            })

            for param_idx, param_name in enumerate(dists_to_calc[dist_name]["params"]):
                
                boostrap_CI = bootstrap(
                    (df[num_var], ),
                    statistic=lambda x: dists_to_calc[dist_name]["dist"].fit(x)[param_idx],
                    confidence_level=0.95,
                    method='BCa'
                )

                stats_dict[num_var][f"{dist_name}_{param_name}_{"low"}"] = round(boostrap_CI.confidence_interval[0], round_to)
                stats_dict[num_var][f"{dist_name}_{param_name}_{"high"}"] = round(boostrap_CI.confidence_interval[1], round_to)

                boostrap_confidences[num_var][f"{dist_name}_{param_name}"] = boostrap_CI.bootstrap_distribution
                
        
    return stats_dict, boostrap_confidences


In [ ]:
def plot_bootstrap_confidences_dist(boostrap_confidences: dict[str, dict[str, np.ndarray]], stats_dict: dict[str, dict[str, float]]) -> None:
    fig, axes = plt.subplots(len(boostrap_confidences), 2*3, figsize = (4 * (2*3), 4 * len(boostrap_confidences)))

    for axs, stat_name in zip(axes, boostrap_confidences):
        for ax, param_name in zip(axs, boostrap_confidences[stat_name]):
            sns.histplot(
                data=boostrap_confidences[stat_name][param_name],
                shrink=0.9,
                ax=ax
            )
            ax.grid(True, ls="--")
            ax.set_title(f"{stat_name}\n{param_name}")

            ax.axvline(stats_dict[stat_name][param_name], ls = "-", lw = 1, label = "stat", color = f"C{1}")

            ax.fill_betweenx(
                [ax.get_ylim()[i] for i in range(2)], 
                stats_dict[stat_name][f"{param_name}_{"low"}"],
                stats_dict[stat_name][f"{param_name}_{"high"}"],
                label = "CI",
                color = f"C{1}",
                alpha = 0.2
            )



    fig.tight_layout()

In [ ]:
dists_to_calc={
    "norm": {"dist": norm, "params": ["mu", "std"]},
    "expon": {"dist": expon, "params": ["loc", "scale"]},
    "logistic": {"dist": logistic, "params": ["loc", "scale"]},
}

stats_dict, boostrap_confidences = calculate_dists_by_boostrap(
    df, numerical_variables,
    dists_to_calc=dists_to_calc
)

Estymowano Parametry i Przedziały Parametrów dla Rozkładów:
- Normalnego (`norm`)
- Wykładniczego (`expon`)
- Logistycznego (`logistic`)

Z Wykresów jasno widać że nie zostały one idealnie dopasowane.

In [ ]:
pd.DataFrame(stats_dict).T

Wartości dla Parametrów i ich przedziały ufności w tabeli, uzyskane za pomocą bootstrap.

In [ ]:
plot_bootstrap_confidences_dist(
    boostrap_confidences, stats_dict
)

Przedziały ufności parametrów z dystrybucji Bootstrap

### Testy Zgodności z rozkładem

In [ ]:
from scipy.stats import kstest, anderson

In [ ]:
def results_to_df(results) -> pd.DataFrame:
    rows = []
    for var, dists in results.items():
        for dist_name, res in dists.items():
            rows.append({
                'variable':  var,
                'dist':      dist_name,
                'kstest_p_value': round(res['kstest_p_value'], 4),
                "anderson_p_value":   round(res["anderson_p_value"], 4),
                'reject H0 from kstest': res['kstest_p_value'] < 0.05,
                'reject H0 from anderson': res['anderson_p_value'] < 0.05,

            })
    return pd.DataFrame(rows).set_index(['dist', 'variable']).sort_index()

def tests_goodness_of_fit(df: pd.DataFrame, dists_to_calc: dict, df_stats: pd.DataFrame) -> pd.DataFrame:
    results = {}

    for var in df_stats.index:
        data = df[var].dropna().values
        results[var] = {}

        for dist_name, dist_info in dists_to_calc.items():
            dist   = dist_info["dist"]
            params = dist_info["params"]

            fitted_params = tuple(
                df_stats.loc[var, f'{dist_name}_{p}']
                for p in params
            )

            kst_stats = kstest(data, dist.cdf, args=fitted_params)

            and_stats = anderson(data, dist=dist_name, method='interpolate')

            results[var][dist_name] = {
                # 'kstest_stat': kst_stats.statistic,
                'kstest_p_value': kst_stats.pvalue,
                # 'anderson_stat': and_stats.statistic,
                "anderson_p_value": and_stats.pvalue,
            }

    return results_to_df(results)

In [ ]:
tests_results = tests_goodness_of_fit(
    df,
    dists_to_calc,
    pd.DataFrame(stats_dict).T
)

tests_results

Testy zgodności z rozkładami (wyżej wymienione) sugerują że Hipoteza Zerowa (rozkład pochodzi z `norm`, `expon`, `logistic`) została odrzucona w prawie każdym przypadku.

Dla reszty przypadków, są one niepewne i raczej wskazują takze na odrzucenie H0, nie mniej dla upewnienia się warto użyć (przynajmniej dla rozkładu normalnego) jego odpowienika, testu Lillieforsa.

## Po Podziale na gatunek

### Estymacja parametrów i Przedziały Ufności

In [ ]:
df["Species"].unique().tolist()

Mamy 3 Gatunki, będziemy testować je oddzielnie

In [ ]:
def test_spcecie(specie_type: Literal['Adelie', 'Chinstrap', 'Gentoo']) -> tuple[pd.DataFrame, pd.DataFrame]:
    stats_dict, boostrap_confidences = calculate_dists_by_boostrap(
        df[df["Species"]==specie_type], numerical_variables,
        dists_to_calc=dists_to_calc
    )

    plot_bootstrap_confidences_dist(boostrap_confidences, stats_dict)

    tests_results = tests_goodness_of_fit(
        df[df["Species"]==specie_type],
        dists_to_calc,
        pd.DataFrame(stats_dict).T
    )


    return pd.DataFrame(stats_dict).T, tests_results

#### Adelie

In [ ]:
df_specie_stats, df_test_results = test_spcecie("Adelie")

Rozkłady na oko wyglądają na dopasowane

In [ ]:
df_specie_stats

Tutaj widzimy niepewności.

Przejdźmy do testów dopasowania do rozkładów

In [ ]:
df_test_results

Rozkłady Wykładniczy zostaje zdecydowanie odrzucony dla `Adelie`.

Najlepiej dopasowany jest rokład Logistyczny oraz Normalny.

#### Chinstrap

In [ ]:
df_specie_stats, df_test_results = test_spcecie("Chinstrap")

In [ ]:
df_specie_stats

In [ ]:
df_test_results

Tutaj najlepiej dopasowuje się rozkład Normalny, następnie logistyczny

#### Gentoo

In [ ]:
df_specie_stats, df_test_results = test_spcecie("Gentoo")

In [ ]:
df_specie_stats

In [ ]:
df_test_results

In [ ]:
df_test_results.loc["expon"]["reject H0 from anderson"].count()

Tak jak powyżej, rozkład wykładniczy został odrzucony, pzoostałe rozkłady sprawują się także nie za dobrze, choć część zostało przyjętych jako odpowiednie.

## Podział ze względu na płeć i gatunek

Będziemy rozważać tylko czy hipoteza 0-owa odrzucona

In [ ]:
all_tested_types: list[dict[str, str]] = []

for specie_type in df["Species"].unique():
    for sex_type in df["Sex"].unique():
        all_tested_types.append({"Species": specie_type, "Sex": sex_type})

all_tested_types

In [ ]:
def test_penguins_for_multiple_types(df: pd.DataFrame, to_test_types: list[dict[str, str]]) -> dict[str, dict[str, int]]:
    tested_types_outputs: dict[str, dict[str, int]] = {}

    for to_test_type in to_test_types:
        type_df = df.copy()

        for column_name, column_value in to_test_type.items():
            type_df = type_df[type_df[column_name] == column_value]


        stats_dict, _ = calculate_dists_by_boostrap(
            type_df, numerical_variables,
            dists_to_calc=dists_to_calc,
            plot=False,
        )


        tests_results = tests_goodness_of_fit(
            type_df,
            dists_to_calc,
            pd.DataFrame(stats_dict).T
        )

        tested_types_name: str = f"{list(to_test_type.values())[0]}-{list(to_test_type.values())[1]}"

        tested_types_outputs[tested_types_name] = {}

        tested_types_outputs[tested_types_name].update({
            f"[{dist_name}] Kolomogorv: H0 rejected": 
            (tests_results.loc[dist_name]["reject H0 from kstest"] == False).sum()
            for dist_name in tests_results.index.get_level_values(0).unique()
        })

        tested_types_outputs[tested_types_name].update({
            f"[{dist_name}] Anderson: H0 rejected": 
            (tests_results.loc[dist_name]["reject H0 from anderson"] == False).sum()
            for dist_name in tests_results.index.get_level_values(0).unique()
        })

        

        


    return tested_types_outputs

In [ ]:
tested_specie_sex = test_penguins_for_multiple_types(df, all_tested_types)

In [ ]:
pd.DataFrame(tested_specie_sex)

Z Wyników widzimy że najlepiej zadziałało dopasowanie do rozkładu Normalnego

(norm vs logistic taki sam dla Kolomorova-Smirnova, jedynie dla Andersona norm vs logistic - lepszy norm)

# Testy na niezależność, korelacje

## Chi Squared dla zmiennych jakościowych

In [ ]:
from scipy.stats import chi2_contingency

In [ ]:
def examine_chi_squared(df: pd.DataFrame, categorical_variables: list[str], round_to: int = 3) -> dict[str, dict[str, float]]:
    chi2_con_dict: dict[str, dict[str, float]] = {}
    for num_var_1 in categorical_variables:
        chi2_con_dict[num_var_1] = {}
        for num_var_2 in categorical_variables:
            chi2_out = chi2_contingency(pd.crosstab(df[num_var_1], df[num_var_2]))

            chi2_con_dict[num_var_1][num_var_2] = round(chi2_out.pvalue, round_to)

    return chi2_con_dict


In [ ]:
chi_sq_con = examine_chi_squared(df, categorical_variables)

In [ ]:
sns.heatmap(pd.DataFrame(chi_sq_con), center=0.05, cmap="coolwarm", annot=True)

p-values dla zmiennych jakościowych na macierzy symetrycznej

Zależne: 

Species - Island, Species Clutch - Completion, Island - Clutch Completion

Nie-Zależne:

Sex - Species, Sex - Island, Sex - Clutch Completion

## ANOVA & Kruskal TESTs

In [169]:
from scipy.stats import  shapiro, normaltest, levene, fligner, kruskal, f_oneway

In [170]:
def examine_ANOVA(
        df: pd.DataFrame, 
        numerical_variables: list[str], 
        categorical_variables: list[str], 
        round_to: int = 3
    ) -> pd.DataFrame:

    records = []
    index = []

    for cat_var in categorical_variables:
        for num_var in numerical_variables:
            groups = [g[num_var].dropna() for _, g in df.groupby(cat_var)]
            residuals = np.concatenate([(g - g.mean()) for g in groups])

            records.append({
                "shapiro p-val":    round(shapiro(residuals).pvalue, round_to),
                "normaltest p-val": round(normaltest(residuals).pvalue, round_to),
                "levene p-val":     round(levene(*groups).pvalue, round_to),
                "fligner p-val":    round(fligner(*groups).pvalue, round_to),
                "kruskal p-val":    round(kruskal(*groups).pvalue, round_to),
                "ANOVA p-val":      round(f_oneway(*groups).pvalue, round_to),
            })
            index.append((cat_var, num_var))

    multi_index = pd.MultiIndex.from_tuples(index, names=["categorical", "numerical"])
    return pd.DataFrame(records, index=multi_index)

In [173]:
ANOVA_df = examine_ANOVA(
    df, 
    numerical_variables,
    categorical_variables,
)

ANOVA_df.style.map(
    lambda x: None if x > 0.05 else 'color: red; font-weight: bold;', subset = ["shapiro p-val", "normaltest p-val", "levene p-val", "fligner p-val"]
).map(
    lambda x: 'color: green; font-weight: bold;' if x < 0.05 else None, subset = ["kruskal p-val", "ANOVA p-val"]
)

[Kurskal] Zmienne Niezależne (Kategorczyna vs Numeryczna):

In [ ]:
ANOVA_df[ANOVA_df["kruskal p-val"] < 0.05].index.to_list()

[('Species', 'Culmen Length (mm)'),
 ('Species', 'Culmen Depth (mm)'),
 ('Species', 'Flipper Length (mm)'),
 ('Species', 'Body Mass (g)'),
 ('Species', 'Delta 15 N (o/oo)'),
 ('Species', 'Delta 13 C (o/oo)'),
 ('Island', 'Culmen Length (mm)'),
 ('Island', 'Culmen Depth (mm)'),
 ('Island', 'Flipper Length (mm)'),
 ('Island', 'Body Mass (g)'),
 ('Island', 'Delta 15 N (o/oo)'),
 ('Island', 'Delta 13 C (o/oo)'),
 ('Clutch Completion', 'Body Mass (g)'),
 ('Clutch Completion', 'Delta 15 N (o/oo)'),
 ('Clutch Completion', 'Delta 13 C (o/oo)'),
 ('Sex', 'Culmen Length (mm)'),
 ('Sex', 'Culmen Depth (mm)'),
 ('Sex', 'Flipper Length (mm)'),
 ('Sex', 'Body Mass (g)')]

[ANOVA] (w dom. Dopuszczone) Zmienne Niezależne (Kategorczyna vs Numeryczna):

In [184]:
ANOVA_df[ANOVA_df[["shapiro p-val", "normaltest p-val", "levene p-val", "fligner p-val"]]
         .gt(0.05).all(axis=1)][ANOVA_df["ANOVA p-val"] < 0.05].index.to_list()

/tmp/ipykernel_21834/2576863475.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ANOVA_df[ANOVA_df[["shapiro p-val", "normaltest p-val", "levene p-val", "fligner p-val"]]


[('Species', 'Culmen Depth (mm)'),
 ('Species', 'Flipper Length (mm)'),
 ('Island', 'Delta 15 N (o/oo)')]

[Kurskal & ANOVA] (w dom. Dopuszczone) Zmienne Niezależne (Kategorczyna vs Numeryczna):

In [185]:
ANOVA_df[ANOVA_df[["shapiro p-val", "normaltest p-val", "levene p-val", "fligner p-val"]]
         .gt(0.05).all(axis=1)][ANOVA_df["ANOVA p-val"] < 0.05][ANOVA_df["kruskal p-val"] < 0.05].index.to_list()

/tmp/ipykernel_21834/1734245078.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ANOVA_df[ANOVA_df[["shapiro p-val", "normaltest p-val", "levene p-val", "fligner p-val"]]
/tmp/ipykernel_21834/1734245078.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ANOVA_df[ANOVA_df[["shapiro p-val", "normaltest p-val", "levene p-val", "fligner p-val"]]


[('Species', 'Culmen Depth (mm)'),
 ('Species', 'Flipper Length (mm)'),
 ('Island', 'Delta 15 N (o/oo)')]